# Bag-of-Words GRPO (word-heteroskedastic, 16 rollouts)

Zero-step-sampling GRPO against the canonical signal-heteroskedastic dataset: per-prompt noise std varies with prompt composition while the global `Corr(signal, target) = corr` is preserved.

The policy is a Gaussian around the model's scalar prediction, $m_\theta(z\mid x) = \mathcal N(f_\theta(x), \sigma^2)$. Rollouts are samples from this policy; rewards are $-(z-y)^2$ against the noisy target; advantages are standardized within each rollout group.

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.grpo import BagOfWordsGRPOConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:1")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/grpo.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsGRPOConfig.get_canonical(
    dataset="word_heteroskedastic",
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-grpo-word-het-example-r16",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
    num_rollouts_per_sample=16,
    gaussian_stdev=1.0,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"SNR halflife (word quantile): {config.data.snr_halflife_in_word_quantile}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")
print(f"Rollouts/sample: {config.num_rollouts_per_sample}")
print(f"Policy stdev:    {config.gaussian_stdev}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Study folder: /home/nlyu/Code/maxrl-statistics/artifacts/bow-grpo-word-het-example-r16/15-words_corr-0.2_len-128_pow-1.0_ar-0.5_hl-0.15
Dataset corr target: 0.2000
SNR halflife (word quantile): 0.15
Backbone lr: 1.230e-03
Head lr:     8.192e-03
Rollouts/sample: 16
Policy stdev:    1.0


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
state.run_training()

grpo epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

grpo epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

Training saves:
1. A compact `metrics.parquet` to disk which contains per-epoch sufficient statistics to compute metrics.
2. Validation parquet containing per-row ground-truth and target.

In [4]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,train_ground_truth_xx,train_ground_truth_xy,train_ground_truth_yy,train_ground_truth_n,val_target_xx,val_target_xy,val_target_yy,val_target_n,val_ground_truth_xx,val_ground_truth_xy,val_ground_truth_yy,val_ground_truth_n
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,7155.380371,965.052551,49942.640625,49984.0,7155.380371,945.21228,2003.919067,49984.0,3586.325684,1207.291016,50076.695312,49920.0,3586.325684,1150.353271,2002.146606,49920.0
1,4666.922363,1664.579956,49945.46875,49984.0,4666.922363,1493.244263,2003.562134,49984.0,3513.658691,1699.139893,50076.695312,49920.0,3513.658691,1610.966797,2002.146606,49920.0


In [5]:
analysis = BagOfWordsAnalysisConfig.from_grouped({
    "example": [(0, config.study_folder)]
})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])

In [6]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
-0.554688,-0.367188,-0.098633
0.025391,0.0859375,-0.933594
-0.077148,0.03125,0.691406
-0.104004,0.0,0.096191
-0.141602,0.015625,-0.429688
